# 🧪 Lab: ใช้ AI ช่วยเชื่อมโยงข้อมูลและทำแผนที่สถานการณ์โรค
### กรณีศึกษา: โรคไข้หวัดใหญ่รายจังหวัด ปี 2568 (ข้อมูลสมมติ)

**เป้าหมาย (40 นาที)**
1. เชื่อมโยงข้อมูล **3 แหล่ง** ได้แก่ ผู้ป่วย ประชากร และเขตสุขภาพ
2. คำนวณ **อัตราป่วยต่อแสนประชากร** แล้วสรุปรายเขตสุขภาพ
3. ทำ **แผนที่แสดงสถานการณ์โรค** รายจังหวัด
4. ลองให้ **Gemini** เขียนโค้ดวิเคราะห์ต่อจากคำสั่งภาษาไทย

**วิธีใช้ Lab นี้**
| สัญลักษณ์ | ความหมาย |
|---|---|
| ▶️ **กดรัน** | คลิกปุ่ม ▶ ที่มุมซ้ายของเซลล์ หรือกด `Shift + Enter` |
| ✨ **ถาม Gemini** | คลิกในเซลล์ว่าง แล้วกด **"generate"** หรือไอคอน ✨ จากนั้นพิมพ์คำสั่งภาษาไทยแล้วกด Enter |
| 🔑 **เฉลย** | เซลล์ที่ซ่อนโค้ดไว้ ถ้า Gemini ตอบไม่ได้ผล กดรันเฉลยได้เลย |

> ⚠️ ข้อมูลใน Lab นี้เป็น **ข้อมูลสมมติเพื่อการสอน** ไม่ใช่สถานการณ์จริง
> ในการทำงานจริง **ห้ามอัปโหลดข้อมูลผู้ป่วยรายบุคคล** ขึ้นเครื่องมือ AI สาธารณะ (พ.ร.บ.คุ้มครองข้อมูลส่วนบุคคล)

**ก่อนเริ่ม:** เมนู **File → Save a copy in Drive** เพื่อให้มีสำเนาของตัวเอง

---
## ส่วนที่ 1 ▶️ เตรียมเครื่องมือและเรียกข้อมูลจาก GitHub
ข้อมูลทั้งหมดเก็บไว้บน GitHub โค้ดจะดึงข้อมูลมาให้อัตโนมัติ **ไม่ต้องอัปโหลดไฟล์เอง**

In [ ]:
#@title ▶️ กดรันเซลล์นี้: เตรียมเครื่องมือ
import pandas as pd
import geopandas as gpd
import folium
import plotly.express as px

BASE = "https://raw.githubusercontent.com/rittiin/ai_workshopsit/main/data/"

pd.set_option("display.max_rows", 100)
print("✅ พร้อมใช้งาน")

In [ ]:
#@title ▶️ กดรันเซลล์นี้: เรียกข้อมูล 3 แหล่ง + ขอบเขตจังหวัด
# dtype={"pro_code": str} = อ่านรหัสจังหวัดเป็นข้อความ เพื่อให้จับคู่กับไฟล์แผนที่ได้
cases  = pd.read_csv(BASE + "influenza_cases_2568.csv", dtype={"pro_code": str})
pop    = pd.read_csv(BASE + "population_2568.csv",      dtype={"pro_code": str})
region = pd.read_csv(BASE + "health_region.csv",        dtype={"pro_code": str})
geo    = gpd.read_file(BASE + "thailand_provinces.geojson")

print(f"แหล่งที่ 1 ผู้ป่วยไข้หวัดใหญ่ : {len(cases)} จังหวัด, ผู้ป่วยรวม {cases['cases'].sum():,} ราย")
print(f"แหล่งที่ 2 ประชากร         : {len(pop)} จังหวัด, ประชากรรวม {pop['population'].sum():,} คน")
print(f"แหล่งที่ 3 เขตสุขภาพ        : {len(region)} จังหวัด, {region['region_no'].nunique()} เขต")
print(f"แผนที่ขอบเขตจังหวัด        : {len(geo)} จังหวัด")

### ดูหน้าตาข้อมูลแต่ละแหล่ง
สังเกตว่าทุกไฟล์มีคอลัมน์ **`pro_code`** (รหัสจังหวัด 2 หลัก) ซึ่งเป็น **"กุญแจ"** ที่ใช้เชื่อมข้อมูล

In [ ]:
#@title ▶️ ดูตัวอย่างข้อมูล
from IPython.display import display
for name, df in [("ผู้ป่วย", cases), ("ประชากร", pop), ("เขตสุขภาพ", region)]:
    print(f"📄 {name}")
    display(df.head(3))

---
## ส่วนที่ 2 ▶️ เชื่อมโยงข้อมูล 3 แหล่ง
เปรียบเหมือนใช้ `VLOOKUP` ใน Excel 2 ครั้ง: นำประชากรและเขตสุขภาพมาต่อท้ายตารางผู้ป่วยโดยใช้ `pro_code`

In [ ]:
#@title ▶️ รวมข้อมูล และตรวจว่าจับคู่ครบทุกจังหวัดไหม
df = (cases
      .merge(pop[["pro_code", "population"]], on="pro_code", how="left", indicator="match_pop")
      .merge(region[["pro_code", "health_region", "region_no"]], on="pro_code", how="left"))

missing = df[df["population"].isna() | df["health_region"].isna()]
print(f"รวมแล้วได้ {len(df)} แถว")
print("✅ จับคู่ครบทุกจังหวัด" if missing.empty else f"⚠️ จับคู่ไม่ได้ {len(missing)} จังหวัด:\n{missing[['pro_code','province']]}")
df = df.drop(columns="match_pop")
df.head()

---
## ส่วนที่ 3 ▶️ คำนวณอัตราป่วยและสรุปรายเขตสุขภาพ
- **อัตราป่วยต่อแสนประชากร** = ผู้ป่วย ÷ ประชากร × 100,000
- **อัตราป่วยตาย (CFR, %)** = ผู้เสียชีวิต ÷ ผู้ป่วย × 100

In [ ]:
#@title ▶️ คำนวณอัตราป่วย และดู 10 จังหวัดสูงสุด
df["rate"] = (df["cases"] / df["population"] * 100_000).round(2)
df["cfr"]  = (df["deaths"] / df["cases"] * 100).round(3)

top10 = df.sort_values("rate", ascending=False).head(10)
top10[["province", "health_region", "cases", "population", "rate"]].reset_index(drop=True)

In [ ]:
#@title ▶️ สรุปรายเขตสุขภาพ
by_region = (df.groupby(["region_no", "health_region"], as_index=False)
               [["cases", "deaths", "population"]].sum())
by_region["rate"] = (by_region["cases"] / by_region["population"] * 100_000).round(2)
by_region.sort_values("region_no").drop(columns="region_no").reset_index(drop=True)

---
## ส่วนที่ 4 ▶️ แผนที่สถานการณ์โรคไข้หวัดใหญ่รายจังหวัด
นำผลลัพธ์ไปเชื่อมกับแผนที่ด้วย `pro_code` เหมือนเดิม **ซูม** ได้ และ **ชี้เมาส์** ที่จังหวัดเพื่อดูตัวเลข

In [ ]:
#@title ▶️ สร้างแผนที่
mapdf = geo.merge(df, on="pro_code", how="left")

# แบ่งสีเป็น 5 ช่วงเท่า ๆ กันตามจำนวนจังหวัด (quantile)
bins = mapdf["rate"].quantile([0, .2, .4, .6, .8, 1]).round(0).tolist()
bins[0], bins[-1] = mapdf["rate"].min() // 1, mapdf["rate"].max() // 1 + 1

m = folium.Map(location=[13.0, 101.0], zoom_start=6, tiles="OpenStreetMap")
folium.Choropleth(
    geo_data=mapdf, data=mapdf, columns=["pro_code", "rate"],
    key_on="feature.properties.pro_code",
    fill_color="YlOrRd", fill_opacity=0.8, line_opacity=0.3,
    bins=bins,
    legend_name="อัตราป่วยไข้หวัดใหญ่ต่อแสนประชากร ปี 2568 (ข้อมูลสมมติ)",
).add_to(m)
folium.GeoJson(
    mapdf, style_function=lambda x: {"fillOpacity": 0, "weight": 0},
    tooltip=folium.GeoJsonTooltip(
        fields=["province", "health_region", "cases", "rate"],
        aliases=["จังหวัด", "เขตสุขภาพ", "ผู้ป่วย (ราย)", "อัตราป่วย/แสน"],
        localize=True),
).add_to(m)
m

In [ ]:
#@title ▶️ (ทางเลือก) บันทึกแผนที่และตารางผลลัพธ์ลงเครื่อง
m.save("flu_map_2568.html")
with pd.ExcelWriter("flu_summary_2568.xlsx") as w:
    df.to_excel(w, sheet_name="รายจังหวัด", index=False)
    by_region.to_excel(w, sheet_name="รายเขตสุขภาพ", index=False)
try:
    from google.colab import files
    files.download("flu_map_2568.html"); files.download("flu_summary_2568.xlsx")
except ImportError:
    print("บันทึกไฟล์แล้ว: flu_map_2568.html, flu_summary_2568.xlsx")

---
## ส่วนที่ 5 ✨ ลองถาม Gemini
ตอนนี้มีตาราง **`df`** (รายจังหวัด) และ **`by_region`** (รายเขตสุขภาพ) อยู่แล้ว ลองสั่ง Gemini ให้เขียนโค้ดต่อ

**วิธีทำ:** คลิกเซลล์ว่างใต้โจทย์ → กด **generate** / ✨ → **คัดลอกคำสั่งไปวาง** → Enter → ตรวจโค้ด → กด ▶️

> 💡 ระบุชื่อตาราง (`df`) และชื่อคอลัมน์ในคำสั่ง จะช่วยให้ Gemini ตอบได้แม่นขึ้น
> คอลัมน์ใน `df`: `pro_code, province, cases, deaths, population, health_region, region_no, rate, cfr`

### โจทย์ 5.1: กราฟแท่ง 10 จังหวัดอัตราป่วยสูงสุด
```
ใช้ตาราง df สร้างกราฟแท่งแนวนอนด้วย plotly แสดง 10 จังหวัดที่มีคอลัมน์ rate สูงสุด เรียงจากมากไปน้อย ใส่ชื่อกราฟภาษาไทย
```

In [ ]:
# ✨ คลิกที่นี่ แล้วกด generate เพื่อพิมพ์คำสั่งกับ Gemini

In [ ]:
#@title 🔑 เฉลย 5.1
fig = px.bar(top10.sort_values("rate"), x="rate", y="province", orientation="h",
             text="rate", color="rate", color_continuous_scale="YlOrRd",
             title="10 จังหวัดอัตราป่วยไข้หวัดใหญ่สูงสุด ปี 2568 (ต่อแสนประชากร)",
             labels={"rate": "อัตราป่วย/แสน", "province": ""})
fig.show()

### โจทย์ 5.2: จัดระดับความเสี่ยง
```
ใช้ตาราง df แบ่งจังหวัดเป็น 3 ระดับตามคอลัมน์ rate ด้วย quantile ได้แก่ ต่ำ ปานกลาง สูง เก็บในคอลัมน์ใหม่ชื่อ risk แล้วแสดงรายชื่อจังหวัดระดับสูงพร้อมเขตสุขภาพ
```

In [ ]:
# ✨ คลิกที่นี่ แล้วกด generate เพื่อพิมพ์คำสั่งกับ Gemini

In [ ]:
#@title 🔑 เฉลย 5.2
df["risk"] = pd.qcut(df["rate"], q=3, labels=["ต่ำ", "ปานกลาง", "สูง"])
print(df["risk"].value_counts())
df[df["risk"] == "สูง"].sort_values("rate", ascending=False)[["province", "health_region", "rate"]].reset_index(drop=True)

### โจทย์ 5.3: แผนที่ระดับความเสี่ยง (ต่อยอดจากโจทย์ 5.2)
```
ใช้ geo ซึ่งเป็น GeoDataFrame ขอบเขตจังหวัด merge กับ df ด้วย pro_code แล้วทำแผนที่ folium ระบายสีตามคอลัมน์ risk โดยต่ำ=เขียว ปานกลาง=เหลือง สูง=แดง และชี้เมาส์แล้วแสดงชื่อจังหวัดกับ rate
```

In [ ]:
# ✨ คลิกที่นี่ แล้วกด generate เพื่อพิมพ์คำสั่งกับ Gemini

In [ ]:
#@title 🔑 เฉลย 5.3
colors = {"ต่ำ": "#4caf50", "ปานกลาง": "#ffc107", "สูง": "#e53935"}
riskmap = geo.merge(df[["pro_code", "province", "rate", "risk"]], on="pro_code")
riskmap["risk"] = riskmap["risk"].astype(str)
m2 = folium.Map(location=[13.0, 101.0], zoom_start=6, tiles="OpenStreetMap")
folium.GeoJson(
    riskmap,
    style_function=lambda f: {"fillColor": colors.get(f["properties"]["risk"], "#ccc"),
                              "color": "#555", "weight": 0.5, "fillOpacity": 0.8},
    tooltip=folium.GeoJsonTooltip(fields=["province", "risk", "rate"],
                                  aliases=["จังหวัด", "ระดับความเสี่ยง", "อัตราป่วย/แสน"]),
).add_to(m2)
m2

### โจทย์ 5.4: ให้ AI ช่วยเขียนสรุปสถานการณ์
เปิด **แผง Gemini** (ไอคอน ✨ มุมขวาบน) แล้วพิมพ์:
```
จากตาราง df และ by_region ช่วยเขียนสรุปสถานการณ์โรคไข้หวัดใหญ่ 3 บรรทัด สำหรับรายงานผู้บริหาร ระบุจังหวัดและเขตสุขภาพที่อัตราป่วยสูงสุด
```
แล้ว **ตรวจตัวเลขที่ AI เขียน** เทียบกับเซลล์เฉลยด้านล่าง ว่าตรงกันไหม

In [ ]:
#@title 🔑 เฉลย 5.4: ตัวเลขสำหรับตรวจทาน
t = df.sort_values("rate", ascending=False).iloc[0]
r = by_region.sort_values("rate", ascending=False).iloc[0]
total_rate = df["cases"].sum() / df["population"].sum() * 100_000
print(f"1) ปี 2568 มีผู้ป่วยไข้หวัดใหญ่ {df['cases'].sum():,} ราย อัตราป่วย {total_rate:.2f} ต่อแสนประชากร เสียชีวิต {df['deaths'].sum()} ราย")
print(f"2) จังหวัดที่อัตราป่วยสูงสุดคือ {t['province']} ({t['rate']:.2f} ต่อแสน) ใน{t['health_region']}")
print(f"3) เขตสุขภาพที่อัตราป่วยสูงสุดคือ {r['health_region']} ({r['rate']:.2f} ต่อแสน)")

---
## 📌 สรุปและข้อควรระวัง
1. **กุญแจเชื่อมข้อมูล** ควรเป็น **รหัส** (เช่น `pro_code`) ไม่ใช่ชื่อ เพราะชื่อมักสะกดไม่ตรงกัน เช่น "กรุงเทพฯ" กับ "กรุงเทพมหานคร"
2. **ตรวจทุกครั้งหลังรวมข้อมูล**: จำนวนแถวครบไหม ยอดรวมตรงกับต้นฉบับไหม
3. **AI เขียนโค้ดได้เร็ว แต่ผิดได้** อ่านโค้ด ตรวจผล และเทียบตัวเลขเสมอ
4. **ข้อมูลส่วนบุคคล**: ใช้กับ AI สาธารณะเฉพาะข้อมูลสรุปที่ไม่ระบุตัวบุคคล
5. **ต่อยอด**: เปลี่ยน `BASE` หรือไฟล์ข้อมูลเป็นโรคอื่นได้ ถ้าไฟล์มีคอลัมน์ `pro_code` เหมือนกัน

---
<sub>ขอบเขตจังหวัด: [OpenGISData-Thailand](https://github.com/chingchai/OpenGISData-Thailand) (CC BY 4.0) ลดความละเอียดเพื่อการสอน ·
ข้อมูลผู้ป่วย/ประชากร: ข้อมูลสมมติ · จัดทำโดย ฤทธิไกร นามเกษ</sub>